## API-Based Model Inference

As an alternative to running models locally, you can also access them via an API. Typically, you then have to pay for input and output tokens, with prices depending on the provider, model, and whether you send requests in batches (cheaper) or separately.

For students and researchers, the Chat AI service of KISSKI/Uni Göttingen is also available for free.

- Test models on the web interface (use federated login): https://chat-ai.academiccloud.de/
- To get an API key, click "Book" here: https://kisski.gwdg.de/en/leistungen/2-02-llm-service/ and fill out the form.
- Manage your API keys here: https://saia.gwdg.de/dashboard
- Available models: https://docs.hpc.gwdg.de/services/ai-services/chat-ai/models/index.html
- API documentation: https://docs.hpc.gwdg.de/services/ai-services/saia/index.html

**Note that there are rate limits** on the API that you have to follow. Also, we had some issues with longer prompts not being properly processed in the past, probably due to an internal context window limit that is not well-documented.

In [ ]:
%%capture
%pip install openai

In [1]:
from openai import OpenAI

# API configuration
api_key = '<key>' # Replace with your API key
base_url = "https://chat-ai.academiccloud.de/v1"

# Start OpenAI client
client = OpenAI(
    api_key = api_key,
    base_url = base_url
)

In [5]:
for model in client.models.list().data:
    print(model.id)

apertus-70b-instruct-2509
devstral-2-123b-instruct-2512
qwen3.8-27b
deepseek-v4-flash-0731
glm-5.3-flash
qwen3-coder-next
qwen3-omni-30b-a3b-instruct
mistral-medium-3.5-128b
qwen3.5-397b-a17b
gemma-4-31b-it
qwen3.6-35b-a3b
meta-llama-3.1-8b-instruct
openai-gpt-oss-120b
qwen3-30b-a3b-instruct-2507


In [6]:
# Get response
chat_completion = client.chat.completions.create(
        messages=[
            {"role":"system","content":"You are a helpful assistant"},
            {"role":"user","content":"How tall is the Eiffel tower?"},
            {"role":"assistant","content":"The Eiffel Tower stands at a height of 324 meters (1,063 feet) above ground level. However, if you include the radio antenna on top, the total height is 330 meters (1,083 feet)."},
            {"role":"user","content":"Are there restaurants?"}
        ],
        model= 'apertus-70b-instruct-2509',
    )

# Print full response as JSON
print(chat_completion) # You can extract the response text from the JSON object

ChatCompletion(id='chatcmpl-808ca2cb58da6f5c', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, there are several restaurants located within the Eiffel Tower:\n\n1. Le 58 Tour Eiffel: Located on the first floor, this restaurant offers traditional French cuisine with spectacular views of the city.\n2. Le Jules Verne: This is a Michelin-starred restaurant on the second floor run by the famous chef Alain Ducasse. It offers gourmet French cuisine with a focus on seasonal and local products.\n3. Bar à Champagne: Located on the top floor, this is a bar that offers champagne and other drinks while you enjoy the stunning views of Paris.\n\nPlease note that dining at the Eiffel Tower requires advanced reservations, and prices can be high due to the exceptional location. Also, the restaurants may have specific hours of operation and there may be availability for reservations according to these hours. Always check ahead before planning a vi

In [10]:
print(chat_completion.choices[0].message.content)

Yes, there are several restaurants located within the Eiffel Tower:

1. Le 58 Tour Eiffel: Located on the first floor, this restaurant offers traditional French cuisine with spectacular views of the city.
2. Le Jules Verne: This is a Michelin-starred restaurant on the second floor run by the famous chef Alain Ducasse. It offers gourmet French cuisine with a focus on seasonal and local products.
3. Bar à Champagne: Located on the top floor, this is a bar that offers champagne and other drinks while you enjoy the stunning views of Paris.

Please note that dining at the Eiffel Tower requires advanced reservations, and prices can be high due to the exceptional location. Also, the restaurants may have specific hours of operation and there may be availability for reservations according to these hours. Always check ahead before planning a visit. 

Additionally, there are cafes located on the ground floor and first floor that offer food and drinks without a need for a reservation. 

Remember t

In [11]:
## A reasoning model
chat_completion = client.chat.completions.create(
        messages=[
            {"role":"system","content":"You are a helpful assistant"},
            {"role":"user","content":"How tall is the Eiffel tower?"},
            {"role":"assistant","content":"The Eiffel Tower stands at a height of 324 meters (1,063 feet) above ground level. However, if you include the radio antenna on top, the total height is 330 meters (1,083 feet)."},
            {"role":"user","content":"Are there restaurants?"}
        ],
        model= 'qwen3.8-27b',
    )

# Print full response as JSON
print(chat_completion)

ChatCompletion(id='chatcmpl-9f87847afccc5756', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="\n\nYes, there are several dining and drink options located inside the Eiffel Tower:\n\n* **Brasserie 1889 (First Floor):** This is the main restaurant in the tower. It offers classic French bistro-style meals, snacks, and drinks while providing a great view of the Parisian street level.\n* **Le Bar (Second Floor):** A relaxed lounge space offering cocktails, wines, champagne, and light bites with a panoramic view of the city.\n* **Le Jule's (Second Floor):** A hidden, speakeasy-style cocktail bar that features a menu of classic and creative cocktails.\n\n**A few helpful tips:**\n* **Tickets are required:** You must have a valid ticket to the specific floor (or a higher floor) to access the restaurants and bars on that level.\n* **Reservations:** While walk-ins are sometimes accommodated, it is highly recommended to book a table in advance,

In [15]:
from IPython.display import display, Markdown

display(Markdown(chat_completion.choices[0].message.reasoning))

Thinking:

1.  **Analyze the Request:**
    *   User asks: "Are there restaurants?"
    *   Context: Previous turn was about the Eiffel Tower. So the implicit subject is "Are there restaurants *in/at* the Eiffel Tower?"
2.  **Retrieve Knowledge (Eiffel Tower Restaurants):**
    *   *Yes*, there are restaurants in the Eiffel Tower.
    *   First floor: Brasserie 1889 (serves bistro-style food, snacks, wine, etc.).
    *   Second floor: Le Bar (drinks, light snacks), and previously Le Jule's (which is actually a speakeasy/cocktail bar, but let's stick to the main restaurants). Wait, Le Jule's is on the 2nd floor.
    *   Top level (Summit): Alouette (Le Cinq is a fine-dining restaurant that *was* on the top, but did it close? Let me check. Actually, Le Cinq was a 3-Michelin-star restaurant at the top. Let me verify if it's still open. Actually, Le Cinq closed or maybe it's still there? Let's be safe. Wait, the Eiffel Tower currently has a restaurant on the second floor called "Le Bar" or "Le Jule's" (cocktail bar) and "Brasserie 1889" on the first floor. There's also a bistro on the top? Let's state the general availability and the most famous one, Brasserie 1889.)
    *   Let's check current status of Le Cinq: Le Cinq at the Eiffel Tower is a famous fine dining restaurant. Wait, actually, I should just state that yes, there are restaurants, specifically on the first floor (Brasserie 1889) and cocktail bars/snack options on the second floor.
    *   Let's refine: Yes, there are restaurants and bars inside the Eiffel Tower.
        *   **First Floor:** Brasserie 1889 (casual dining, bistro style).
        *   **Second Floor:** Le Bar (drinks and light bites) and Le Jule's (a speakeasy-style cocktail bar).
        *   *(Self-Correction/Refinement on Le Cinq)*: Le Cinq was historically at the summit. I will mention the first and second floor ones to be perfectly accurate, as Brasserie 1889 is the main full-service restaurant currently widely advertised.
3.  **Formulate the Output:**
    *   Direct answer: Yes, there are restaurants and bars inside the Eiffel Tower.
    *   Details:
        *   **Brasserie 1889** (First Floor): Offers bistro-style meals, snacks, and drinks.
        *   **Le Bar** (Second Floor): A lounge area for drinks and light snacks.
        *   **Le Jule's** (Second Floor): A speakeasy-style cocktail bar.
    *   Note: Reservations are highly recommended, especially for the first floor, and they can only be reached by stairs or elevator depending on the ticket.
4.  **Review against constraints:**
    *   Helpful, clear, concise. No markdown issues.
5.  **Draft Response:**
    Yes, there are restaurants and bars located inside the Eiffel Tower:

    *   **Brasserie 1889 (First Floor):** This is the main restaurant in the tower, offering classic French bistro-style dishes, snacks, and drinks.
    *   **Le Bar (Second Floor):** A relaxed lounge space offering cocktails, wines, and light bites with a view.
    *   **Le Jule's (Second Floor):** A hidden, speakeasy-style cocktail bar.

    *Note: Reservations for the restaurants and bars are highly recommended, as they can fill up quickly, especially during peak tourist seasons.* (Wait, can you book Le Jule's? Yes, but keep it general: "Reservations for the dining options are highly recommended...")

    Let's double check if there's a restaurant at the top. Le Cinq was at the top. Actually, it might be closed or not open to the general public the same way. I will stick to the 1st and 2nd floor to be safe and accurate. Brasserie 1889 is definitely the primary one.

    Let's write out the final response clearly.
    - Yes, there are.
    - Brasserie 1889 (1st floor)
    - Le Bar (2nd floor)
    - Le Jule's (2nd floor)
    - Tip: Book in advance. You need a ticket to the specific floor to access the restaurants. (Actually, to get to the 2nd floor restaurant, you need a 2nd floor or summit ticket. To get to 1st floor, just a 1st floor ticket or higher). Keep it simple. Just answer the question.
    - "Yes, there are several dining and drink options inside the Eiffel Tower..." followed by the list. Good. All facts are solid. No hallucinations. (Brasserie 1889 is open to the public with a ticket). All good. Let's output.
    *Wait, did Le Cinq close? Yes, Le Cinq closed in 2020 or so, or maybe it's still there? Actually, I don't need to mention the top to avoid inaccuracy. The 1st and 2nd floor are 100% correct.*
    *Let's just state the facts.*
    - Brasserie 1889 (1st Floor)
    - Le Bar (2nd Floor)
    - Le Jule's (2nd Floor)
    Perfect. I will format nicely.
    End of thought process.


In [16]:
display(Markdown(chat_completion.choices[0].message.content))



Yes, there are several dining and drink options located inside the Eiffel Tower:

* **Brasserie 1889 (First Floor):** This is the main restaurant in the tower. It offers classic French bistro-style meals, snacks, and drinks while providing a great view of the Parisian street level.
* **Le Bar (Second Floor):** A relaxed lounge space offering cocktails, wines, champagne, and light bites with a panoramic view of the city.
* **Le Jule's (Second Floor):** A hidden, speakeasy-style cocktail bar that features a menu of classic and creative cocktails.

**A few helpful tips:**
* **Tickets are required:** You must have a valid ticket to the specific floor (or a higher floor) to access the restaurants and bars on that level.
* **Reservations:** While walk-ins are sometimes accommodated, it is highly recommended to book a table in advance, especially for Brasserie 1889 during peak tourist seasons and summer months.